# Sensitivity analysis: limiting dam-station links by drainage-area ratio and distance

**Purpose.** `link_dams_stations.py` links every station anywhere downstream of a removed dam, with no distance or size limit. The median downstream pair is about 55-60 km from its dam, and its station drains about 48x the dam's area, so a small dam's signal is mostly diluted by other water. This notebook asks: **if links were limited to stations that mostly carry the dam's water, does the estimated before/after change get clearer, stay the same, or disappear?**

**Drainage-area ratio (DAR)** = station upstream area / dam upstream area (NHDPlus `TotDASqKm`, from `pair_features.parquet` written by `preprocessing_audit.ipynb`). DAR ≈ 1 means the station sees mostly the dam's water; DAR = 50 means the dam drains ~2% of it. DAR only applies to downstream stations.

**How to read the result:**
- An effect that holds or grows as the limits tighten, and differs from the upstream reference, is plausibly a removal effect.
- An effect that shrinks to zero or flips sign as the limits tighten was likely regional trend or dilution, not the dam.
- Wider intervals at tight limits mean fewer pairs, not necessarily no effect; check the sample-size table.

**Steps**
1. Load data and check DAR coverage
2. Per-pair change (daily values, seasonal cycle removed)
3. Competing removals, recomputed within each distance limit
4. Threshold grid: mean change per series with dam-level bootstrap CIs
5. Sample size per threshold
6. Plots: change vs threshold, against an upstream reference

Exploratory only. Nothing in the warehouse or ETL is changed.

**Since 2026-10-26** the pipeline applies the chosen limit (config `exposure: max_downstream_km`, 10 km), so `analysis_measurement` no longer has far pairs. To re-run this grid, build the views and analysis dataset with both limits removed from the config (`build_views` then `build_analysis_dataset`; the covariate stages too, if new stations become eligible).

In [ ]:
from pathlib import Path
import os
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from etl.common import wq_rules
from etl.common.config import load_config
from etl.common.warehouse import warehouse_dir

# Where data/ lives: the repo on the cluster (DAM_REMOVAL_PROJECT_ROOT overrides it, as in the ETL).
DATA_ROOT = Path(os.environ.get("DAM_REMOVAL_PROJECT_ROOT", ROOT))
CASE_STUDY = "all_removals"
cfg = load_config(CASE_STUDY, project_root=DATA_ROOT)
WAREHOUSE = warehouse_dir(cfg)
AUDIT_DIR = DATA_ROOT / "data" / "processed" / "preprocessing_audit" / CASE_STUDY
OUTPUT_DIR = DATA_ROOT / "data" / "processed" / "dar_sensitivity" / CASE_STUDY

window = cfg.get("wq_window", {}) or {}
YEARS_BEFORE = int(window.get("years_before", 3))
YEARS_AFTER = int(window.get("years_after", 3))

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", 30)

In [ ]:
# ---- Settings: the grid to test, not decisions ----
DAR_LIMITS = [1.5, 2, 5, 10, None]       # downstream station area / dam area; None = no limit
DISTANCE_LIMITS_KM = [5, 10, 25, None]   # network distance from the dam; None = no limit
SERIES_OF_INTEREST = ["water_temp", "dissolved_oxygen", "ph", "specific_conductance", "turbidity",
                      "phosphorus (Total)", "orthophosphate (Dissolved)", "tds"]
MIN_PAIRS = 10    # cells with fewer downstream pairs aren't estimated
N_BOOT = 1000     # bootstrap resamples of dams
SEED = 0

# Skewed, always-positive parameters are compared on log10 (as in analysis_eda.ipynb).
LOG_PARAMETERS = {"turbidity", "specific_conductance", "nitrate", "ammonia",
                  "orthophosphate", "phosphorus", "tds"}

DAR_LABELS = [f"≤{d:g}" if d is not None else "none" for d in DAR_LIMITS]
DIST_LABELS = [f"≤{d:g} km" if d is not None else "none" for d in DISTANCE_LIMITS_KM]


def y_label(series):
    parameter = series.split(" (")[0]
    unit = wq_rules.STANDARD_UNITS.get(parameter, "")
    return f"log10 {unit}" if parameter in LOG_PARAMETERS else unit


def facet_axes(n, ncols=4, width=4, height=3):
    """A grid of n axes, extra ones hidden."""
    nrows = -(-n // ncols)
    fig, axes = plt.subplots(nrows, ncols, figsize=(width * ncols, height * nrows), squeeze=False)
    for ax in axes.flat[n:]:
        ax.set_visible(False)
    return fig, list(axes.flat)

## 1. Load data and check DAR coverage

`analysis_measurement` has one row per measurement per eligible dam-station pair. DAR comes from the audit's `pair_features.parquet`. Pairs missing from that file (e.g. if the audit was run on an older build) or without an NHDPlus area can't be filtered by DAR. They are left out of the whole grid, including the "none" baseline, so every cell compares the same pool of pairs.

Rows dated in the future (a known WQP entry error, see `warehouse_qa`) are dropped here as a guard.

In [ ]:
df = pd.read_parquet(WAREHOUSE / "analysis_measurement.parquet", columns=[
    "measurement_id", "AR_ID", "station_id", "parameter_std", "sample_fraction_std", "removal_year",
    "side", "network_distance_km", "period", "sample_date", "month", "result_value_std"])

future = pd.to_datetime(df["sample_date"]) > pd.Timestamp.today()
print(f"Dropped {int(future.sum()):,} rows dated in the future")
df = df[~future]

df["series"] = df["parameter_std"].where(
    df["sample_fraction_std"].isna(),
    df["parameter_std"] + " (" + df["sample_fraction_std"].fillna("") + ")",
)
df = df[df["series"].isin(SERIES_OF_INTEREST)]
df["y"] = np.where(df["parameter_std"].isin(LOG_PARAMETERS),
                   np.log10(df["result_value_std"].clip(lower=1e-3)), df["result_value_std"])
df["pair"] = df["AR_ID"] + " | " + df["station_id"]

features = pd.read_parquet(AUDIT_DIR / "pair_features.parquet", columns=[
    "pair", "drainage_area_ratio", "ratio_status", "upstream_class"])

# One row per pair, with its DAR status.
pairs = (df[["pair", "AR_ID", "station_id", "side", "removal_year", "network_distance_km"]]
         .drop_duplicates("pair")
         .merge(features, on="pair", how="left", indicator=True))
pairs["dar_status"] = np.where(pairs["_merge"] == "left_only", "not in pair_features",
                               pairs["ratio_status"])
pairs = pairs.drop(columns="_merge")

print(f"Rows: {len(df):,}; pairs: {len(pairs):,}; dams: {pairs['AR_ID'].nunique()}")
display(pd.crosstab(pairs["dar_status"], pairs["side"], margins=True))

down_pairs = pairs[(pairs["side"] == "downstream") & (pairs["dar_status"] == "ok")]
print("Downstream pairs with a valid DAR:")
down_pairs[["drainage_area_ratio", "network_distance_km"]].describe(
    percentiles=[0.1, 0.25, 0.5, 0.75, 0.9]).round(1)

## 2. Per-pair change

The same method as section 5 of `analysis_eda.ipynb`, with one change:

1. **Daily values.** Readings are first reduced to one median per station, series and day, so a continuous logger doesn't outweigh monthly grab samples.
2. **Anomaly.** Each daily value minus the station's mean for that series and calendar month, over both periods. This removes the station's level and its seasonal cycle.
3. **Change.** For each pair and series: mean anomaly after removal minus mean anomaly before. Units are those of y (log10 for skewed parameters, so 0.05 ≈ +12%).

One number per pair keeps heavily sampled pairs from dominating; pairs are then averaged with equal weight.

In [ ]:
# 1. One value per station, series and day.
measurements = df.drop_duplicates("measurement_id")
daily = (measurements.groupby(["station_id", "series", "sample_date"], as_index=False)
         .agg(y=("y", "median"), month=("month", "first")))

# 2. Departure from the station's mean for that series and calendar month.
daily["anomaly"] = daily["y"] - daily.groupby(["station_id", "series", "month"])["y"].transform("mean")

# 3. A station-day can belong to several pairs (one per linked dam), each with its own period.
pair_days = (df[["pair", "station_id", "series", "sample_date", "period"]].drop_duplicates()
             .merge(daily[["station_id", "series", "sample_date", "anomaly"]],
                    on=["station_id", "series", "sample_date"]))
change = pair_days.groupby(["pair", "series", "period"])["anomaly"].mean().unstack("period")
change = change.dropna(subset=["pre", "post"])
change["change"] = change["post"] - change["pre"]
change = (change.reset_index()[["pair", "series", "change"]]
          .merge(pairs[["pair", "AR_ID", "side", "dar_status", "upstream_class"]], on="pair"))

print(f"Pair-series changes: {len(change):,}")
change.groupby(["series", "side"])["change"].describe()[["count", "50%"]].unstack("side").round(3)

## 3. Competing removals within the limits

A pair "competes" when its station also sits downstream of another removed dam whose removal falls inside this pair's window, so the change can't be attributed to one dam. In the audit this was computed over all links. Here it is recomputed **within each distance limit**: a dam 300 km upstream no longer counts as a competitor once links are capped at 25 km.

The other dam's DAR isn't known (`pair_features` only covers eligible pairs), so competitors are limited by distance only.

In [ ]:
links = pd.read_parquet(WAREHOUSE / "dam_station_link.parquet",
                        columns=["AR_ID", "station_id", "relationship", "network_distance_km"])
removal_years = pd.read_parquet(WAREHOUSE / "dam_removal.parquet", columns=["AR_ID", "Year_Removed"])
# Removed dams each station sits downstream of.
competitors = (links[links["relationship"] == "downstream"]
               .merge(removal_years.dropna(subset=["Year_Removed"]), on="AR_ID"))


def competing(subset, distance_limit):
    """True for pairs whose station is also downstream of another dam removed inside this
    pair's window, within distance_limit km (None = any distance)."""
    other = competitors if distance_limit is None else \
        competitors[competitors["network_distance_km"] <= distance_limit]
    x = subset[["pair", "AR_ID", "station_id", "removal_year"]].merge(
        other[["AR_ID", "station_id", "Year_Removed"]], on="station_id", suffixes=("", "_other"))
    x = x[(x["AR_ID_other"] != x["AR_ID"])
          & x["Year_Removed"].between(x["removal_year"] - YEARS_BEFORE, x["removal_year"] + YEARS_AFTER)]
    return subset["pair"].isin(x["pair"])


for limit, label in zip(DISTANCE_LIMITS_KM, DIST_LABELS):
    print(f"Distance {label}: {competing(down_pairs, limit).mean():.0%} of downstream pairs have a competing removal")

## 4. Threshold grid

For every DAR limit × distance limit, keep the downstream pairs inside both limits, then estimate the mean per-pair change for each series. Two versions:

- **all pairs**: every pair inside the limits
- **no competing removal**: also drops pairs with a competing removal (section 3)

The 95% interval comes from resampling **dams** with replacement. Pairs of one dam aren't independent, so resampling pairs or rows would give intervals that are too narrow (the problem with the preview models in `analysis_eda`).

The **upstream reference** is the same per-pair change at upstream stations, excluding those possibly in the former reservoir. It approximates what changed regionally over the same years without the removal. A downstream change no different from it is weak evidence of a removal effect.

In [ ]:
rng = np.random.default_rng(SEED)


def dam_bootstrap(rows):
    """Mean per-pair change, with a 95% CI from resampling dams."""
    by_dam = rows.groupby("AR_ID")["change"].agg(["sum", "count"])
    sums, counts = by_dam["sum"].to_numpy(), by_dam["count"].to_numpy()
    idx = rng.integers(0, len(by_dam), size=(N_BOOT, len(by_dam)))
    boot = sums[idx].sum(axis=1) / counts[idx].sum(axis=1)
    low, high = np.percentile(boot, [2.5, 97.5])
    return rows["change"].mean(), low, high


down_change = change[(change["side"] == "downstream") & (change["dar_status"] == "ok")]
records, coverage = [], []
for dist, dist_label in zip(DISTANCE_LIMITS_KM, DIST_LABELS):
    for dar, dar_label in zip(DAR_LIMITS, DAR_LABELS):
        kept = down_pairs
        if dar is not None:
            kept = kept[kept["drainage_area_ratio"] <= dar]
        if dist is not None:
            kept = kept[kept["network_distance_km"] <= dist]
        kept = kept.assign(competing=competing(kept, dist))
        for version, subset in [("all pairs", kept), ("no competing removal", kept[~kept["competing"]])]:
            coverage.append({"version": version, "distance_limit": dist_label, "dar_limit": dar_label,
                             "dams (pairs)": f"{subset['AR_ID'].nunique()} ({len(subset)})"})
            for series, rows in down_change[down_change["pair"].isin(subset["pair"])].groupby("series"):
                record = {"version": version, "distance_limit": dist_label, "dar_limit": dar_label,
                          "series": series, "pairs": len(rows), "dams": rows["AR_ID"].nunique()}
                if len(rows) >= MIN_PAIRS:
                    record["mean_change"], record["ci_low"], record["ci_high"] = dam_bootstrap(rows)
                records.append(record)
results = pd.DataFrame(records)

# Upstream reference: no limits (DAR doesn't apply upstream), former-reservoir stations excluded.
upstream = change[(change["side"] == "upstream")
                  & (change["upstream_class"] != "possibly in former reservoir")]
upstream_ref = {series: dam_bootstrap(rows) for series, rows in upstream.groupby("series")
                if len(rows) >= MIN_PAIRS}
print(f"{len(results):,} grid cells; {int(results['mean_change'].notna().sum()):,} with >= {MIN_PAIRS} pairs")

## 5. Sample size

Dams (pairs) left in each cell, for any series. The bottom-right cell of the "all pairs" table (no limits) is the current analysis set, restricted to pairs with a valid DAR. Tight limits that leave only a handful of dams can't support any conclusion, whatever the plots show.

In [ ]:
coverage = pd.DataFrame(coverage)
for version, table in coverage.groupby("version"):
    print(version)
    display(table.pivot(index="distance_limit", columns="dar_limit", values="dams (pairs)")
            .reindex(index=DIST_LABELS, columns=DAR_LABELS))

## 6. Change vs threshold

One panel per series. x: DAR limit (tightest left). Colors: distance limit. Points: mean per-pair change after − before, with the 95% dam-bootstrap interval. Gray band and dotted line: the upstream reference.

What to look for:
- **Consistent direction as limits tighten** (moving left, and toward ≤5 km): plausibly a removal effect. Expected directions for small-dam removals are cooler water and more DO near the dam; nutrients and turbidity are mixed and often short-lived.
- **The effect shrinks toward 0 or flips as limits tighten**: the unrestricted estimate was likely regional trend or dilution.
- **Downstream overlaps the upstream band**: no evidence the downstream change differs from what happened regionally.

In [ ]:
def plot_version(version):
    data = results[(results["version"] == version) & results["mean_change"].notna()]
    fig, axes = facet_axes(len(SERIES_OF_INTEREST))
    offsets = np.linspace(-0.24, 0.24, len(DIST_LABELS))
    for ax, series in zip(axes, SERIES_OF_INTEREST):
        if series in upstream_ref:
            mean, low, high = upstream_ref[series]
            ax.axhspan(low, high, color="gray", alpha=0.2, lw=0)
            ax.axhline(mean, color="gray", ls=":", lw=1)
        ax.axhline(0, color="k", lw=0.8)
        for offset, dist_label, color in zip(offsets, DIST_LABELS, sns.color_palette(n_colors=len(DIST_LABELS))):
            rows = data[(data["series"] == series) & (data["distance_limit"] == dist_label)]
            if rows.empty:
                continue
            x = [DAR_LABELS.index(label) + offset for label in rows["dar_limit"]]
            ax.errorbar(x, rows["mean_change"],
                        yerr=[rows["mean_change"] - rows["ci_low"], rows["ci_high"] - rows["mean_change"]],
                        fmt="o", ms=4, capsize=2, color=color, label=dist_label)
        ax.set_xticks(range(len(DAR_LABELS)))
        ax.set_xticklabels(DAR_LABELS)
        ax.set(title=series, xlabel="DAR limit", ylabel=f"change ({y_label(series)})")
    axes[0].legend(title="distance limit", fontsize=8)
    fig.suptitle(f"Downstream per-pair change vs link limits ({version}); gray = upstream reference")
    fig.tight_layout()
    plt.show()


plot_version("all pairs")

In [ ]:
plot_version("no competing removal")

## 7. Results table and summary

The full grid is saved to `data/processed/dar_sensitivity/<case study>/dar_sensitivity_results.csv`. The table below shows the headline cells: no limit vs a moderate limit (DAR ≤ 2, ≤ 10 km), without competing removals.

In [ ]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results.to_csv(OUTPUT_DIR / "dar_sensitivity_results.csv", index=False)

headline = results[(results["version"] == "no competing removal")
                   & (((results["dar_limit"] == "none") & (results["distance_limit"] == "none"))
                      | ((results["dar_limit"] == "≤2") & (results["distance_limit"] == "≤10 km")))]
(headline.assign(limits=headline["dar_limit"] + ", " + headline["distance_limit"])
 .pivot(index="series", columns="limits", values=["mean_change", "ci_low", "ci_high", "dams"])
 .round(3))

**Limitations**
- DAR is an area ratio, not measured flow. It is less reliable where runoff varies a lot across the watershed (arid, snowmelt-fed or diverted rivers).
- Competing removals are limited by distance only (no DAR for non-eligible dams), and only *removed* dams are known: dams still standing between the station and the removal aren't.
- The anomaly removes level and season, not long-term trends or flow. The upstream reference is a rough control for trends, not a matched one.
- Censored values are used as stored. Stations shared by several dams are still counted once per dam in the bootstrap.
- `pair_features.parquet` must come from the same build as `analysis_measurement` (see the coverage table in section 1).